# 07 — BERTopic: Failure Pattern Discovery

Goal: Discover latent failure themes without using labels.

Expected finding:
- BERTopic should find ~8–15 coherent topics across 3,000 work orders
- Topics should broadly align with the 6 labeled categories
- But some topics will reveal sub-themes not explicitly labeled:
  e.g., 'seal leaks from pressure spikes' vs. 'seal leaks from fluid contamination'

Interview story:
> 'BERTopic found sub-themes I hadn't explicitly labeled — it separated seal leaks
> from two different root causes. Those need different corrective actions. An unsupervised
> model found domain structure I know exists but didn't encode explicitly.'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import sys; sys.path.insert(0, '..')
from src.nlp_pipeline import preprocess_series

%matplotlib inline

In [ ]:
df = pd.read_csv('../data/work_orders.csv')
docs = df.text.tolist()
labels_true = df.failure_category.tolist()

# Optionally use preprocessed text
# docs = preprocess_series(df.text).tolist()

In [ ]:
# Fit BERTopic
# Note: first run downloads sentence-transformer model (~90 MB)
topic_model = BERTopic(
    embedding_model='all-MiniLM-L6-v2',
    min_topic_size=30,
    nr_topics='auto',
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs)
print(f'Discovered {len(set(topics)) - 1} topics (excluding outlier topic -1)')

In [ ]:
# Topic overview
topic_info = topic_model.get_topic_info()
print(topic_info.head(20).to_string())
# Look for: are seal/bearing topics split by mechanism?

In [ ]:
# Visualize topics
topic_model.visualize_topics()

In [ ]:
# Compare BERTopic topics to known labels
cross = pd.crosstab(
    pd.Series(labels_true, name='true_category'),
    pd.Series(topics, name='bertopic_cluster')
)
print(cross.to_string())
# Expect each true category to map to 1–3 BERTopic clusters (sub-themes)

In [ ]:
# Save topic model for reference
topic_model.save('../models/bertopic_model')
print('BERTopic model saved to models/bertopic_model')
print('Proceed to notebook 08 for similarity search.')